# Airbnb Listing Price Prediction: Berlin, Paris and NYC

Requires listings_detail_berlin.csv, listings_pari26.csv and listings_nyc26.csv in Google Drive at MyDrive/airbnb_project/data/raw/

## Berlin baselines


In [1]:
# Cell 1 — Setup and load Berlin dataset

import pandas as pd
import numpy as np

from sklearn.utils import shuffle

from google.colab import drive
drive.mount('/content/gdrive')

# Load Berlin Airbnb data
df = pd.read_csv(
    '/content/gdrive/MyDrive/airbnb_project/data/raw/listings_detail_berlin.csv'
)

# Shuffle the dataset
df = shuffle(df, random_state=42).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))

Mounted at /content/gdrive
Dataset shape: (14274, 79)
Number of columns: 79


In [2]:
# Cell 2 — Check the available columns

print(df.columns.tolist())

['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 'description', 'neighborhood_overview', 'picture_url', 'host_id', 'host_url', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url', 'host_picture_url', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'calendar_updated', 'has_availability', 'availability_30', 'availability_60', 'availability_90', 'availabil

In [3]:
# Cell 3 — Inspect price values

print(df['price'].head(10))
print("\nMissing prices:", df['price'].isna().sum())
print("\nPrice data type:", df['price'].dtype)

0    $175.00
1     $90.00
2    $269.00
3    $122.00
4    $225.00
5        NaN
6    $225.00
7    $201.00
8        NaN
9        NaN
Name: price, dtype: object

Missing prices: 5010

Price data type: object


In [4]:
# Cell 4 — Clean price and remove listings without a usable price

df = df.dropna(subset=['price']).copy()

df['price'] = (
    df['price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
)

df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Remove anything that still couldn't be converted
df = df.dropna(subset=['price']).copy()

print("Dataset shape after removing missing prices:", df.shape)
print("Price data type:", df['price'].dtype)
print("Minimum price:", df['price'].min())
print("Maximum price:", df['price'].max())

Dataset shape after removing missing prices: (9264, 79)
Price data type: float64
Minimum price: 5.0
Maximum price: 50000.0


In [5]:
# Cell 5 — Inspect price distribution

print(df['price'].describe())

print("\nSelected price percentiles:")
print(df['price'].quantile([0.50, 0.75, 0.90, 0.95, 0.99, 0.995, 0.999]))

count     9264.000000
mean       201.240393
std       1656.989769
min          5.000000
25%         70.000000
50%        104.000000
75%        160.000000
max      50000.000000
Name: price, dtype: float64

Selected price percentiles:
0.500      104.00
0.750      160.00
0.900      238.00
0.950      300.00
0.990      617.37
0.995      999.00
0.999    40000.00
Name: price, dtype: float64


In [6]:
# Cell 6 — Apply the price threshold used in the original report

price_thres = 500

df = df[df['price'] <= price_thres].copy()

print("Dataset shape after price filtering:", df.shape)
print("Maximum price:", df['price'].max())
print("Number of listings removed:", 9264 - len(df))

Dataset shape after price filtering: (9133, 79)
Maximum price: 500.0
Number of listings removed: 131


In [7]:
# Cell 7 — Inspect column data types

print(df.dtypes.to_string())

id                                                int64
listing_url                                      object
scrape_id                                         int64
last_scraped                                     object
source                                           object
name                                             object
description                                      object
neighborhood_overview                            object
picture_url                                      object
host_id                                           int64
host_url                                         object
host_name                                        object
host_since                                       object
host_location                                    object
host_about                                       object
host_response_time                               object
host_response_rate                               object
host_acceptance_rate                            

In [8]:
# Cell 8 — Remove identifiers and URL metadata

drop_cols = [
    'id',
    'host_id',
    'listing_url',
    'scrape_id',
    'picture_url',
    'host_url',
    'host_thumbnail_url',
    'host_picture_url',
    'calendar_updated'
]

df = df.drop(columns=drop_cols, errors='ignore')

print("Shape after dropping metadata:", df.shape)

Shape after dropping metadata: (9133, 70)


In [9]:
# Cell 9 — Check missing values in numerical columns

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

missing_numeric = df[numeric_cols].isna().sum()
missing_numeric = missing_numeric[missing_numeric > 0].sort_values(ascending=False)

print(missing_numeric)

reviews_per_month              2018
review_scores_checkin          2018
review_scores_value            2018
review_scores_location         2018
review_scores_communication    2018
review_scores_accuracy         2018
review_scores_cleanliness      2018
review_scores_rating           2018
bedrooms                         46
beds                              9
maximum_minimum_nights            5
minimum_minimum_nights            5
maximum_maximum_nights            5
minimum_maximum_nights            5
bathrooms                         5
host_listings_count               2
host_total_listings_count         2
dtype: int64


In [10]:
# Cell 10 — Fill missing numerical values with their median

numeric_feature_cols = [
    col for col in numeric_cols
    if col != 'price'
]

for col in numeric_feature_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

print("Remaining numerical missing values:",
      df[numeric_feature_cols].isna().sum().sum())

Remaining numerical missing values: 0


In [11]:
# Cell 11 — Check missing categorical values

categorical_cols_all = df.select_dtypes(include=['object']).columns.tolist()

missing_categorical = df[categorical_cols_all].isna().sum()
missing_categorical = missing_categorical[missing_categorical > 0].sort_values(ascending=False)

print(missing_categorical)

host_neighbourhood        5970
neighborhood_overview     5025
neighbourhood             5025
host_about                4134
host_location             2217
first_review              2018
last_review               2018
license                   1895
host_response_rate        1152
host_response_time        1152
host_acceptance_rate      1014
description                339
host_is_superhost          140
has_availability           132
bathrooms_text               7
host_name                    2
host_since                   2
host_verifications           2
host_has_profile_pic         2
host_identity_verified       2
dtype: int64


In [12]:
# Cell 12 — Fill missing categorical values

for col in categorical_cols_all:
    df[col] = df[col].fillna('Unknown')

print("Remaining categorical missing values:",
      df[categorical_cols_all].isna().sum().sum())

Remaining categorical missing values: 0


In [13]:
# Cell 13 — Identify text features

text_cols = [
    'name',
    'description',
    'neighborhood_overview',
    'host_about',
    'amenities'
]

print("Text columns:")
for col in text_cols:
    print("-", col)

Text columns:
- name
- description
- neighborhood_overview
- host_about
- amenities


In [14]:
# Cell 14 — Prepare text data

for col in text_cols:
    df[col] = df[col].astype(str)

print("Text data prepared.")
print(df[text_cols].head(2))

Text data prepared.
                                                name  \
0         Blueground | Prenzlauer Berg, nr Mauerpark   
1  La Petite Maison Balkon Checkpoint Alexanderplatz   

                                         description  \
0  Discover the best of Berlin, with this one bed...   
1  Discover Berlin with all its facets and come w...   

                               neighborhood_overview  \
0  This furnished apartment is located in Prenzla...   
1  All shops serving daily needs are 50-250 m awa...   

                                          host_about  \
0  We’re Blueground, a global proptech company wi...   
1  Ich habe 3 Kinder. Daher habe ich selbst gerne...   

                                           amenities  
0  ["Cooking basics", "Crib", "Shampoo", "Lockbox...  
1  ["Cooking basics", "Wine glasses", "Crib", "Dr...  


In [15]:
# Cell 15 — TF-IDF for listing names

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_name = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

name_tfidf = tfidf_name.fit_transform(df['name'])

print("Name TF-IDF shape:", name_tfidf.shape)

Name TF-IDF shape: (9133, 4383)


In [16]:
# Cell 16 — SVD for listing names

from sklearn.decomposition import TruncatedSVD

svd_name = TruncatedSVD(n_components=50, random_state=42)

name_svd = svd_name.fit_transform(name_tfidf)

print("Name SVD shape:", name_svd.shape)

Name SVD shape: (9133, 50)


In [17]:
# Cell 17 — TF-IDF for descriptions

tfidf_description = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

description_tfidf = tfidf_description.fit_transform(df['description'])

print("Description TF-IDF shape:", description_tfidf.shape)

Description TF-IDF shape: (9133, 5000)


In [18]:
# Cell 18 — SVD for descriptions

svd_description = TruncatedSVD(n_components=50, random_state=42)

description_svd = svd_description.fit_transform(description_tfidf)

print("Description SVD shape:", description_svd.shape)

Description SVD shape: (9133, 50)


In [19]:
# Cell 19 — TF-IDF for neighborhood_overview

tfidf_neighborhood = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

neighborhood_tfidf = tfidf_neighborhood.fit_transform(
    df['neighborhood_overview']
)

print("Neighborhood overview TF-IDF shape:", neighborhood_tfidf.shape)

Neighborhood overview TF-IDF shape: (9133, 5000)


In [20]:
# Cell 20 — SVD for neighborhood_overview

svd_neighborhood = TruncatedSVD(n_components=50, random_state=42)

neighborhood_svd = svd_neighborhood.fit_transform(neighborhood_tfidf)

print("Neighborhood overview SVD shape:", neighborhood_svd.shape)

Neighborhood overview SVD shape: (9133, 50)


In [21]:
# Cell 21 — TF-IDF for host_about

tfidf_host_about = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

host_about_tfidf = tfidf_host_about.fit_transform(
    df['host_about']
)

print("Host about TF-IDF shape:", host_about_tfidf.shape)

Host about TF-IDF shape: (9133, 5000)


In [22]:
# Cell 22 — SVD for host_about

svd_host_about = TruncatedSVD(n_components=50, random_state=42)

host_about_svd = svd_host_about.fit_transform(host_about_tfidf)

print("Host about SVD shape:", host_about_svd.shape)

Host about SVD shape: (9133, 50)


In [23]:
# Cell 23 — TF-IDF for amenities

tfidf_amenities = TfidfVectorizer(
    max_features=5000,
    stop_words='english'
)

amenities_tfidf = tfidf_amenities.fit_transform(
    df['amenities']
)

print("Amenities TF-IDF shape:", amenities_tfidf.shape)

Amenities TF-IDF shape: (9133, 1213)


In [24]:
# Cell 24 — SVD for amenities

svd_amenities = TruncatedSVD(n_components=50, random_state=42)

amenities_svd = svd_amenities.fit_transform(amenities_tfidf)

print("Amenities SVD shape:", amenities_svd.shape)

Amenities SVD shape: (9133, 50)


In [25]:
# Cell 25 — Check date columns

date_cols = [
    'last_scraped',
    'host_since',
    'first_review',
    'last_review',
    'calendar_last_scraped'
]

for col in date_cols:
    print(f"\n{col}:")
    print(df[col].head(3).tolist())


last_scraped:
['2025-09-24', '2025-09-24', '2025-09-24']

host_since:
['2021-12-03', '2016-04-12', '2023-11-13']

first_review:
['Unknown', '2017-05-15', '2023-12-12']

last_review:
['Unknown', '2025-06-27', '2025-09-08']

calendar_last_scraped:
['2025-09-24', '2025-09-24', '2025-09-24']


In [26]:
# Cell 26 — Convert dates to numerical features

from datetime import date

date_cols = ['host_since', 'first_review', 'last_review']

def clean_date(cols):
    dates = pd.to_datetime(cols, errors='coerce')

    # Find the earliest valid date
    min_date = pd.Timestamp(date.today())
    for col in dates:
        if pd.notna(col):
            min_date = min(min_date, col)

    # Mean date is used for missing values
    mean_date = dates.mean()

    # Difference from earliest date, measured in hours
    dif = (mean_date - min_date).total_seconds() / 3600

    arr = np.zeros((len(cols), 1))

    for i, col in enumerate(dates):
        if pd.isna(col):
            arr[i] = dif
        else:
            arr[i] = (col - min_date).total_seconds() / 3600

    return arr

date_features = []

for col in date_cols:
    date_features.append(clean_date(df[col]))

date_features = np.concatenate(date_features, axis=1)

print("Date features shape:", date_features.shape)
print("First 5 rows:")
print(date_features[:5])

Date features shape: (9133, 3)
First 5 rows:
[[116760.         106560.16528461 111176.94560787]
 [ 67296.          69264.         113688.        ]
 [133800.         126912.         115440.        ]
 [ 58992.         135264.         115416.        ]
 [133800.         129000.         114672.        ]]


/tmp/ipykernel_3660/1913670262.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(cols, errors='coerce')
/tmp/ipykernel_3660/1913670262.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dates = pd.to_datetime(cols, errors='coerce')


In [27]:
# Cell 27 — Define categorical features

one_hot_col_names = [
    'host_response_time',
    'host_is_superhost',
    'host_has_profile_pic',
    'host_identity_verified',
    'neighbourhood_cleansed',
    'neighbourhood_group_cleansed',
    'property_type',
    'room_type',
    'has_availability',
    'instant_bookable',
    'cancellation_policy'
]

print("Categorical columns:")
for col in one_hot_col_names:
    print("-", col)

Categorical columns:
- host_response_time
- host_is_superhost
- host_has_profile_pic
- host_identity_verified
- neighbourhood_cleansed
- neighbourhood_group_cleansed
- property_type
- room_type
- has_availability
- instant_bookable
- cancellation_policy


In [28]:
# Cell 28 — One-hot encoding function

from sklearn.preprocessing import LabelEncoder, OneHotEncoder

def one_hot(arr):
    label_encoder = LabelEncoder()
    integer_encoded = label_encoder.fit_transform(arr)

    onehot_encoder = OneHotEncoder(sparse_output=False)
    integer_encoded = integer_encoded.reshape(len(integer_encoded), 1)

    onehot_encoded = onehot_encoder.fit_transform(integer_encoded)

    return onehot_encoded

print("One-hot encoding function ready.")

One-hot encoding function ready.


In [29]:
# Cell 30 — Remove unavailable categorical column

one_hot_col_names = [
    col for col in one_hot_col_names
    if col in df.columns
]

print("Categorical columns being used:")
print(one_hot_col_names)

Categorical columns being used:
['host_response_time', 'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'property_type', 'room_type', 'has_availability', 'instant_bookable']


In [30]:
# Cell 29 — One-hot encode categorical columns

one_hot_dict = {}

for col in one_hot_col_names:
    one_hot_dict[col] = one_hot(
        np.array(df[col].fillna(""), dtype=str)
    )
    print(col, "->", one_hot_dict[col].shape)

print("\nNormal categorical encoding complete.")

host_response_time -> (9133, 5)
host_is_superhost -> (9133, 3)
host_has_profile_pic -> (9133, 3)
host_identity_verified -> (9133, 3)
neighbourhood_cleansed -> (9133, 138)
neighbourhood_group_cleansed -> (9133, 12)
property_type -> (9133, 63)
room_type -> (9133, 4)
has_availability -> (9133, 2)
instant_bookable -> (9133, 2)

Normal categorical encoding complete.


In [31]:
# Cell 31 — Inspect host verification values

print(df['host_verifications'].head(10).tolist())

["['email', 'phone']", "['email', 'phone', 'work_email']", "['email', 'phone']", "['email', 'phone', 'work_email']", "['email', 'phone']", "['email', 'phone', 'work_email']", "['email', 'phone']", "['email', 'phone']", "['email', 'phone']", "['email', 'phone']"]


In [32]:
# Cell 32 — Encode host verifications

def encode_host_verifications(cols):
    all_values = []

    for value in cols:
        try:
            items = eval(value)
        except:
            items = []

        all_values.extend(items)

    unique_values = sorted(set(all_values))

    encoded = np.zeros((len(cols), len(unique_values)))

    for i, value in enumerate(cols):
        try:
            items = eval(value)
        except:
            items = []

        for item in items:
            if item in unique_values:
                encoded[i, unique_values.index(item)] = 1

    return encoded, unique_values


host_verification_features, host_verification_names = encode_host_verifications(
    df['host_verifications']
)

print("Host verification feature shape:", host_verification_features.shape)
print("Verification types:", host_verification_names)

Host verification feature shape: (9133, 3)
Verification types: ['email', 'phone', 'work_email']


In [33]:
# Cell 33 — Inspect amenities values

print(df['amenities'].head(5).tolist())

['["Cooking basics", "Crib", "Shampoo", "Lockbox", "Private entrance", "High chair", "Patio or balcony", "Self check-in", "Heating", "TV", "Bed linens", "Oven", "Kitchen", "Iron", "Hot water", "Wifi", "Smoke alarm", "Washer \\u2013\\u00a0In unit", "Microwave", "Essentials", "Dishwasher", "Long term stays allowed", "Pets allowed", "Hair dryer", "Dryer \\u2013\\u00a0In unit", "Hangers", "Refrigerator"]', '["Cooking basics", "Wine glasses", "Crib", "Drying rack for clothing", "Free dryer \\u2013 In unit", "Hot water kettle", "Baking sheet", "Elevator", "High chair", "Free washer \\u2013 In unit", "Books and reading material", "Cleaning products", "Dedicated workspace", "Bauknecht refrigerator", "Bed linens", "Outdoor dining area", "Courtyard view", "Oven", "Toaster", "Stainless steel electric stove", "Central heating", "Kitchen", "Host greets you", "Iron", "Paid street parking off premises", "Hot water", "Wifi", "Smoke alarm", "Coffee", "Microwave", "Shared backyard \\u2013 Not fully fenc

In [34]:
# Cell 34 — Encode amenities

import ast

def encode_amenities(cols):
    all_values = []

    for value in cols:
        try:
            items = ast.literal_eval(value)
        except:
            items = []

        all_values.extend(items)

    unique_values = sorted(set(all_values))

    encoded = np.zeros((len(cols), len(unique_values)))

    for i, value in enumerate(cols):
        try:
            items = ast.literal_eval(value)
        except:
            items = []

        for item in items:
            if item in unique_values:
                encoded[i, unique_values.index(item)] = 1

    return encoded, unique_values


amenities_features, amenity_names = encode_amenities(df['amenities'])

print("Amenities feature shape:", amenities_features.shape)
print("Number of unique amenities:", len(amenity_names))
print("First 10 amenities:", amenity_names[:10])

Amenities feature shape: (9133, 2392)
Number of unique amenities: 2392
First 10 amenities: [' BEKO refrigerator', ' Bauknecht oven', '(vorhanden) body soap', '1 inch TV with Amazon Prime Video', '100 inch HDTV with Amazon Prime Video, Disney+, Netflix, Apple TV', '100 inch TV with Amazon Prime Video, Netflix', '108 inch TV', '110 inch HDTV with standard cable', '110 inch TV with Fire TV, Netflix, standard cable', '120 L refrigerator']


In [35]:
# Cell 35 — Convert host response rate to numerical values

host_response_rate = (
    df['host_response_rate']
    .astype(str)
    .str.replace('%', '', regex=False)
)

host_response_rate = pd.to_numeric(
    host_response_rate,
    errors='coerce'
)

host_response_rate = host_response_rate.fillna(
    host_response_rate.mean()
)

host_response_rate = host_response_rate.to_numpy().reshape(-1, 1)

print("Host response rate shape:", host_response_rate.shape)
print("Minimum:", host_response_rate.min())
print("Maximum:", host_response_rate.max())
print("Mean:", host_response_rate.mean())

Host response rate shape: (9133, 1)
Minimum: 0.0
Maximum: 100.0
Mean: 90.39230672848015


In [36]:
# Cell 36 — Select continuous features

continuous_cols = [
    'host_listings_count',
    'host_total_listings_count',
    'latitude',
    'longitude',
    'accommodates',
    'bathrooms',
    'bedrooms',
    'beds',
    'minimum_nights',
    'maximum_nights',
    'availability_30',
    'availability_60',
    'availability_90',
    'availability_365',
    'number_of_reviews',
    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value',
    'calculated_host_listings_count',
    'reviews_per_month'
]

continuous_cols = [
    col for col in continuous_cols
    if col in df.columns
]

continuous_features = df[continuous_cols].to_numpy()

print("Number of continuous features:", len(continuous_cols))
print("Continuous feature shape:", continuous_features.shape)
print("\nFeatures used:")
print(continuous_cols)

Number of continuous features: 24
Continuous feature shape: (9133, 24)

Features used:
['host_listings_count', 'host_total_listings_count', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'maximum_nights', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'number_of_reviews', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'calculated_host_listings_count', 'reviews_per_month']


In [37]:
# Cell 37 — Combine categorical features

categorical_features = np.concatenate(
    [one_hot_dict[col] for col in one_hot_col_names]
    + [host_verification_features]
    + [amenities_features],
    axis=1
)

print("Categorical feature shape:", categorical_features.shape)

Categorical feature shape: (9133, 2630)


In [38]:
# Cell 38 — Combine all feature groups

text_features = np.concatenate(
    [
        name_svd,
        description_svd,
        neighborhood_svd,
        host_about_svd
    ],
    axis=1
)

all_features = np.concatenate(
    [
        continuous_features,
        host_response_rate,
        date_features,
        categorical_features,
        text_features
    ],
    axis=1
)

print("Continuous features:", continuous_features.shape[1])
print("Host response rate:", host_response_rate.shape[1])
print("Date features:", date_features.shape[1])
print("Categorical features:", categorical_features.shape[1])
print("Text features:", text_features.shape[1])

print("\nFinal feature matrix shape:", all_features.shape)

Continuous features: 24
Host response rate: 1
Date features: 3
Categorical features: 2630
Text features: 200

Final feature matrix shape: (9133, 2858)


In [39]:
# Cell 39 — Create log-price target

labels = np.log(df['price'].to_numpy())

print("Target shape:", labels.shape)
print("Minimum log(price):", labels.min())
print("Maximum log(price):", labels.max())
print("First 5 target values:", labels[:5])

Target shape: (9133,)
Minimum log(price): 1.6094379124341003
Maximum log(price): 6.214608098422191
First 5 target values: [5.16478597 4.49980967 5.59471138 4.80402104 5.4161004 ]


In [40]:
# Cell 40 — Train / development / test split

num_data = len(all_features)

train_num = int(num_data * 0.7)
dev_num = int(num_data * 0.9)

train_features = all_features[:train_num]
dev_features = all_features[train_num:dev_num]
test_features = all_features[dev_num:]

train_labels = labels[:train_num]
dev_labels = labels[train_num:dev_num]
test_labels = labels[dev_num:]

print("Train:", train_features.shape, train_labels.shape)
print("Dev:  ", dev_features.shape, dev_labels.shape)
print("Test: ", test_features.shape, test_labels.shape)

Train: (6393, 2858) (6393,)
Dev:   (1826, 2858) (1826,)
Test:  (914, 2858) (914,)


In [41]:
# Cell 41 — Normalize using training statistics

mean = np.mean(train_features, axis=0)
std = np.std(train_features, axis=0)

# Prevent division by zero for constant features
std[std == 0] = 1

train_features = (train_features - mean) / std
dev_features = (dev_features - mean) / std
test_features = (test_features - mean) / std

print("Normalization complete.")
print("Train mean (first 5 features):", np.mean(train_features, axis=0)[:5])
print("Train std (first 5 features):", np.std(train_features, axis=0)[:5])

Normalization complete.
Train mean (first 5 features): [-1.34510670e-16  4.40114514e-17 -1.31992418e-11  4.88543283e-15
  1.30854536e-16]
Train std (first 5 features): [1. 1. 1. 1. 1.]


In [42]:
# Cell 42 — Linear Regression

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

linear_model = LinearRegression()

linear_model.fit(train_features, train_labels)

dev_predictions = linear_model.predict(dev_features)

linear_mse = mean_squared_error(dev_labels, dev_predictions)
linear_r2 = r2_score(dev_labels, dev_predictions)

print("Linear Regression")
print("Dev MSE:", linear_mse)
print("Dev R²:", linear_r2)

Linear Regression
Dev MSE: 0.15205571591755457
Dev R²: 0.550316861014372


In [43]:
# Cell 43 — KNN Regression

from sklearn.neighbors import KNeighborsRegressor

knn_model = KNeighborsRegressor(n_neighbors=5)

knn_model.fit(train_features, train_labels)

knn_predictions = knn_model.predict(dev_features)

knn_mse = mean_squared_error(dev_labels, knn_predictions)
knn_r2 = r2_score(dev_labels, knn_predictions)

print("KNN Regression")
print("Dev MSE:", knn_mse)
print("Dev R²:", knn_r2)

KNN Regression
Dev MSE: 0.22407639635696577
Dev R²: 0.3373259487264356


In [44]:
# Cell 44 — Random Forest Regression

from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(train_features, train_labels)

rf_predictions = rf_model.predict(dev_features)

rf_mse = mean_squared_error(dev_labels, rf_predictions)
rf_r2 = r2_score(dev_labels, rf_predictions)

print("Random Forest Regression")
print("Dev MSE:", rf_mse)
print("Dev R²:", rf_r2)

Random Forest Regression
Dev MSE: 0.10780503806780047
Dev R²: 0.6811819429196682


In [45]:
# Cell 45 — XGBoost Regression

from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

xgb_model.fit(train_features, train_labels)

xgb_predictions = xgb_model.predict(dev_features)

xgb_mse = mean_squared_error(dev_labels, xgb_predictions)
xgb_r2 = r2_score(dev_labels, xgb_predictions)

print("XGBoost Regression")
print("Dev MSE:", xgb_mse)
print("Dev R²:", xgb_r2)

XGBoost Regression
Dev MSE: 0.08866230947541896
Dev R²: 0.7377938383043798


In [46]:
# Cell 46 — Neural Network setup

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Input features:", train_features.shape[1])

Device: cuda
Input features: 2858


In [47]:
# Cell 47 — Neural Network architecture

class AirbnbNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x)


model = AirbnbNN(train_features.shape[1]).to(device)

print(model)

AirbnbNN(
  (network): Sequential(
    (0): Linear(in_features=2858, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=512, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=512, out_features=64, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.2, inplace=False)
    (9): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [48]:
# Cell 48 — Create PyTorch DataLoaders

X_train = torch.tensor(train_features, dtype=torch.float32)
y_train = torch.tensor(train_labels, dtype=torch.float32).reshape(-1, 1)

X_dev = torch.tensor(dev_features, dtype=torch.float32)
y_dev = torch.tensor(dev_labels, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train, y_train)
dev_dataset = TensorDataset(X_dev, y_dev)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=64,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Development batches:", len(dev_loader))

Training batches: 100
Development batches: 29


In [49]:
# Cell 49 — Configure optimizer and loss

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate: 1e-4")
print("Weight decay: 1e-5")

Loss function: MSELoss()
Optimizer: Adam
Learning rate: 1e-4
Weight decay: 1e-5


In [50]:
# Cell 50 — Train Neural Network

num_epochs = 100
best_dev_loss = float('inf')
best_model_state = None

train_losses = []
dev_losses = []

for epoch in range(num_epochs):

    # ---- Training ----
    model.train()
    running_train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * X_batch.size(0)

    train_loss = running_train_loss / len(train_loader.dataset)

    # ---- Development evaluation ----
    model.eval()
    running_dev_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in dev_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)

            running_dev_loss += loss.item() * X_batch.size(0)

    dev_loss = running_dev_loss / len(dev_loader.dataset)

    train_losses.append(train_loss)
    dev_losses.append(dev_loss)

    # Save best model
    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        best_model_state = {
            key: value.cpu().clone()
            for key, value in model.state_dict().items()
        }

    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch {epoch + 1:3d}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Dev Loss: {dev_loss:.4f}"
        )

print("\nTraining complete.")
print("Best Dev Loss:", best_dev_loss)

Epoch  10/100 | Train Loss: 0.3816 | Dev Loss: 0.3406
Epoch  20/100 | Train Loss: 0.3369 | Dev Loss: 0.3653
Epoch  30/100 | Train Loss: 0.3071 | Dev Loss: 0.3695
Epoch  40/100 | Train Loss: 0.2902 | Dev Loss: 0.3888
Epoch  50/100 | Train Loss: 0.2626 | Dev Loss: 0.3486
Epoch  60/100 | Train Loss: 0.2557 | Dev Loss: 0.4153
Epoch  70/100 | Train Loss: 0.2406 | Dev Loss: 0.3956
Epoch  80/100 | Train Loss: 0.2375 | Dev Loss: 0.4253
Epoch  90/100 | Train Loss: 0.2224 | Dev Loss: 0.4353
Epoch 100/100 | Train Loss: 0.2167 | Dev Loss: 0.4306

Training complete.
Best Dev Loss: 0.3050800221283621


In [51]:
# Cell 51 — Restore the best Neural Network

model.load_state_dict(best_model_state)
model = model.to(device)
model.eval()

print("Best model restored.")
print("Best development loss:", best_dev_loss)

Best model restored.
Best development loss: 0.3050800221283621


In [52]:
# Cell 52 — Linear Regression test evaluation

linear_test_predictions = linear_model.predict(test_features)

linear_test_mse = mean_squared_error(test_labels, linear_test_predictions)
linear_test_r2 = r2_score(test_labels, linear_test_predictions)

print("Linear Regression — Test")
print("Test MSE:", linear_test_mse)
print("Test R²:", linear_test_r2)

Linear Regression — Test
Test MSE: 0.14479472623881948
Test R²: 0.5779014569704315


In [53]:
# Cell 53 — KNN test evaluation

knn_test_predictions = knn_model.predict(test_features)

knn_test_mse = mean_squared_error(test_labels, knn_test_predictions)
knn_test_r2 = r2_score(test_labels, knn_test_predictions)

print("KNN Regression — Test")
print("Test MSE:", knn_test_mse)
print("Test R²:", knn_test_r2)

KNN Regression — Test
Test MSE: 0.2407979577111816
Test R²: 0.2980375062366355


In [54]:
# Cell 54 — Random Forest test evaluation

rf_test_predictions = rf_model.predict(test_features)

rf_test_mse = mean_squared_error(test_labels, rf_test_predictions)
rf_test_r2 = r2_score(test_labels, rf_test_predictions)

print("Random Forest — Test")
print("Test MSE:", rf_test_mse)
print("Test R²:", rf_test_r2)

Random Forest — Test
Test MSE: 0.1120410911452256
Test R²: 0.6733832608389316


In [55]:
# Cell 55 — XGBoost test evaluation

xgb_test_predictions = xgb_model.predict(test_features)

xgb_test_mse = mean_squared_error(test_labels, xgb_test_predictions)
xgb_test_r2 = r2_score(test_labels, xgb_test_predictions)

print("XGBoost — Test")
print("Test MSE:", xgb_test_mse)
print("Test R²:", xgb_test_r2)

XGBoost — Test
Test MSE: 0.09887582302081609
Test R²: 0.711762010108717


In [56]:
# Cell 56 — Neural Network test evaluation

X_test = torch.tensor(test_features, dtype=torch.float32)
y_test = torch.tensor(test_labels, dtype=torch.float32).reshape(-1, 1)

model.eval()

with torch.no_grad():
    nn_test_predictions = model(X_test.to(device)).cpu().numpy().flatten()

nn_test_mse = mean_squared_error(test_labels, nn_test_predictions)
nn_test_r2 = r2_score(test_labels, nn_test_predictions)

print("Neural Network — Test")
print("Test MSE:", nn_test_mse)
print("Test R²:", nn_test_r2)

Neural Network — Test
Test MSE: 0.3036281852868229
Test R²: 0.11487788290786649


In [57]:
# Cell 57 — Store Berlin baseline results

berlin_results = {
    "Linear Regression": {
        "MSE": linear_test_mse,
        "R2": linear_test_r2
    },
    "KNN": {
        "MSE": knn_test_mse,
        "R2": knn_test_r2
    },
    "Random Forest": {
        "MSE": rf_test_mse,
        "R2": rf_test_r2
    },
    "XGBoost": {
        "MSE": xgb_test_mse,
        "R2": xgb_test_r2
    },
    "Neural Network": {
        "MSE": nn_test_mse,
        "R2": nn_test_r2
    }
}

print("Berlin baseline results saved.")
print(berlin_results)

Berlin baseline results saved.
{'Linear Regression': {'MSE': 0.14479472623881948, 'R2': 0.5779014569704315}, 'KNN': {'MSE': 0.2407979577111816, 'R2': 0.2980375062366355}, 'Random Forest': {'MSE': 0.1120410911452256, 'R2': 0.6733832608389316}, 'XGBoost': {'MSE': 0.09887582302081609, 'R2': 0.711762010108717}, 'Neural Network': {'MSE': 0.3036281852868229, 'R2': 0.11487788290786649}}


## Paris baselines


In [58]:
# Cell 86 — Load Paris dataset

df_paris = pd.read_csv('/content/gdrive/MyDrive/airbnb_project/data/raw/listings_pari26.csv')

print("Paris dataset loaded!")
print("Shape:", df_paris.shape)
print("Price sample:")
print(df_paris['price'].head())

Paris dataset loaded!
Shape: (38075, 90)
Price sample:
0     $85.67
1    $559.50
2    $581.67
3        NaN
4    $799.00
Name: price, dtype: object


In [59]:
# Cell 87 — Clean Paris prices

df_paris['price'] = (
    df_paris['price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

print("Price converted successfully!")
print("Missing prices:", df_paris['price'].isna().sum())
print("Price range:", df_paris['price'].min(), "to", df_paris['price'].max())

Price converted successfully!
Missing prices: 2127
Price range: 11.55 to 70080.0


In [60]:
# Cell 88 — Clean Paris price data

df_paris = df_paris.dropna(subset=['price']).reset_index(drop=True)

# Same price threshold used for Berlin
df_paris = df_paris[df_paris['price'] <= 500].reset_index(drop=True)

print("Paris shape after price cleaning:", df_paris.shape)
print("Price range:", df_paris['price'].min(), "to", df_paris['price'].max())

Paris shape after price cleaning: (30971, 90)
Price range: 11.55 to 500.0


In [61]:
# Cell 89 — Remove metadata columns

drop_cols = [
    'id',
    'listing_url',
    'scrape_id',
    'last_scraped',
    'source',
    'picture_url',
    'host_url',
    'host_thumbnail_url',
    'host_picture_url'
]

existing_drop_cols = [c for c in drop_cols if c in df_paris.columns]

df_paris = df_paris.drop(columns=existing_drop_cols)

print("Paris shape after removing metadata:", df_paris.shape)

Paris shape after removing metadata: (30971, 81)


In [62]:
# Cell 90 — Check Paris feature columns

check_cols = [
    'host_response_time',
    'host_response_rate',
    'host_is_superhost',
    'host_has_profile_pic',
    'host_identity_verified',
    'neighbourhood_cleansed',
    'neighbourhood_group_cleansed',
    'zipcode',
    'is_location_exact',
    'property_type',
    'room_type',
    'bed_type',
    'has_availability',
    'requires_license',
    'instant_bookable',
    'is_business_travel_ready',
    'cancellation_policy',
    'require_guest_profile_picture',
    'require_guest_phone_verification',
    'host_verifications',
    'amenities',
    'host_since',
    'first_review',
    'last_review'
]

for col in check_cols:
    print(f"{col}: {'YES' if col in df_paris.columns else 'NO'}")

host_response_time: YES
host_response_rate: YES
host_is_superhost: YES
host_has_profile_pic: YES
host_identity_verified: YES
neighbourhood_cleansed: YES
neighbourhood_group_cleansed: YES
zipcode: NO
is_location_exact: NO
property_type: YES
room_type: YES
bed_type: NO
has_availability: YES
requires_license: NO
instant_bookable: YES
is_business_travel_ready: NO
cancellation_policy: NO
require_guest_profile_picture: NO
require_guest_phone_verification: NO
host_verifications: YES
amenities: YES
host_since: YES
first_review: YES
last_review: YES


In [63]:
# Cell 91 — Define Paris feature groups

continuous_cols = [
    'host_listings_count',
    'host_total_listings_count',
    'latitude',
    'longitude',
    'accommodates',
    'bathrooms',
    'bedrooms',
    'beds',
    'minimum_nights',
    'maximum_nights',
    'availability_30',
    'availability_60',
    'availability_90',
    'availability_365',
    'number_of_reviews',
    'review_scores_rating',
    'review_scores_accuracy',
    'review_scores_cleanliness',
    'review_scores_checkin',
    'review_scores_communication',
    'review_scores_location',
    'review_scores_value',
    'calculated_host_listings_count',
    'reviews_per_month'
]

categorical_cols = [
    'host_response_time',
    'host_is_superhost',
    'host_has_profile_pic',
    'host_identity_verified',
    'neighbourhood_cleansed',
    'neighbourhood_group_cleansed',
    'property_type',
    'room_type',
    'has_availability',
    'instant_bookable'
]

date_cols = [
    'host_since',
    'first_review',
    'last_review'
]

text_cols = [
    'name',
    'description',
    'neighborhood_overview',
    'host_about'
]

print("Continuous:", len(continuous_cols))
print("Categorical:", len(categorical_cols))
print("Dates:", len(date_cols))
print("Text:", len(text_cols))

Continuous: 24
Categorical: 10
Dates: 3
Text: 4


In [64]:
# Cell 92 — Check missing values in continuous features

for col in continuous_cols:
    print(f"{col}: {df_paris[col].isna().sum()} missing")

host_listings_count: 440 missing
host_total_listings_count: 30971 missing
latitude: 0 missing
longitude: 0 missing
accommodates: 0 missing
bathrooms: 11 missing
bedrooms: 103 missing
beds: 308 missing
minimum_nights: 3 missing
maximum_nights: 3 missing
availability_30: 0 missing
availability_60: 0 missing
availability_90: 0 missing
availability_365: 0 missing
number_of_reviews: 0 missing
review_scores_rating: 3844 missing
review_scores_accuracy: 3844 missing
review_scores_cleanliness: 3844 missing
review_scores_checkin: 3844 missing
review_scores_communication: 3844 missing
review_scores_location: 3844 missing
review_scores_value: 3844 missing
calculated_host_listings_count: 0 missing
reviews_per_month: 3844 missing


In [65]:
# Cell 93 — Handle missing continuous values

# Drop the completely empty column
continuous_cols.remove('host_total_listings_count')
df_paris = df_paris.drop(columns=['host_total_listings_count'])

# Fill remaining missing continuous values with the median
for col in continuous_cols:
    df_paris[col] = df_paris[col].fillna(df_paris[col].median())

print("Continuous features remaining:", len(continuous_cols))
print("Remaining missing values:",
      df_paris[continuous_cols].isna().sum().sum())

Continuous features remaining: 23
Remaining missing values: 0


In [66]:
# Cell 94 — Fill missing categorical values

for col in categorical_cols:
    df_paris[col] = df_paris[col].fillna('Unknown')

print("Remaining categorical missing values:",
      df_paris[categorical_cols].isna().sum().sum())

Remaining categorical missing values: 0


In [67]:
# Cell 95 — Check date columns

for col in date_cols:
    print(f"\n{col}")
    print("Missing:", df_paris[col].isna().sum())
    print("Sample:", df_paris[col].dropna().head(3).tolist())


host_since
Missing: 30971
Sample: []

first_review
Missing: 3844
Sample: ['2025-11-19', '2020-12-13', '2015-12-22']

last_review
Missing: 3844
Sample: ['2026-03-22', '2021-03-08', '2025-11-10']


In [68]:
# Cell 96 — Convert date columns into numeric features

# Drop completely empty date column
date_cols.remove('host_since')
df_paris = df_paris.drop(columns=['host_since'])

# Convert remaining dates to datetime
for col in date_cols:
    df_paris[col] = pd.to_datetime(df_paris[col], errors='coerce')

# Use the earliest valid date as the reference
all_dates = pd.concat([df_paris[col] for col in date_cols]).dropna()
min_date = all_dates.min()

# Convert each date into number of days since min_date
for col in date_cols:
    date_values = (df_paris[col] - min_date).dt.days

    # Replace missing dates with the mean
    date_values = date_values.fillna(date_values.mean())

    df_paris[col] = date_values

print("Date features converted successfully!")
print("Date columns:", date_cols)
print("Missing date values:",
      df_paris[date_cols].isna().sum().sum())

Date features converted successfully!
Date columns: ['first_review', 'last_review']
Missing date values: 0


In [69]:
# Cell 97 — Check missing text values

for col in text_cols:
    print(f"{col}: {df_paris[col].isna().sum()} missing")

name: 0 missing
description: 1048 missing
neighborhood_overview: 30971 missing
host_about: 17101 missing


In [70]:
# Cell 98 — Fill missing text values

for col in text_cols:
    df_paris[col] = df_paris[col].fillna('')

print("Remaining missing text values:",
      df_paris[text_cols].isna().sum().sum())

Remaining missing text values: 0


In [71]:
# Cell 99 — TF-IDF + SVD for available Paris text

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# Only use text columns that contain actual text
usable_text_cols = ['name', 'description', 'host_about']

text_features = []

for col in usable_text_cols:
    print("Processing:", col)

    vectorizer = TfidfVectorizer(
        max_features=5000,
        stop_words='english'
    )

    tfidf_matrix = vectorizer.fit_transform(df_paris[col])

    svd = TruncatedSVD(n_components=50, random_state=42)
    reduced = svd.fit_transform(tfidf_matrix)

    text_features.append(reduced)

text_features = np.concatenate(text_features, axis=1)

print("Final text feature shape:", text_features.shape)

Processing: name
Processing: description
Processing: host_about
Final text feature shape: (30971, 150)


In [72]:
# Cell 100 — One-hot encode Paris categorical features

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

categorical_features = encoder.fit_transform(
    df_paris[categorical_cols]
)

print("Categorical feature shape:", categorical_features.shape)

Categorical feature shape: (30971, 86)


In [73]:
# Cell 101 — Encode host_verifications

verification_values = set()

for value in df_paris['host_verifications']:
    if isinstance(value, str):
        for item in value.split():
            item = item.strip("[]'\",")
            if item:
                verification_values.add(item)

verification_values = sorted(verification_values)

verification_features = np.zeros(
    (len(df_paris), len(verification_values))
)

for i, value in enumerate(df_paris['host_verifications']):
    if isinstance(value, str):
        items = set(item.strip("[]'\",") for item in value.split())
        for j, verification in enumerate(verification_values):
            if verification in items:
                verification_features[i, j] = 1

print("Host verification features:", verification_features.shape)
print("Verification types:", verification_values)

Host verification features: (30971, 0)
Verification types: []


In [74]:
# Cell 102 — Inspect Paris amenities

print(df_paris['amenities'].head(5).tolist())

['["Essentials"]', '["Washer", "Hot water", "Elevator", "Kitchen", "Central heating", "TV", "Wifi"]', '["Washer", "Kitchen", "First aid kit", "TV", "Wifi", "Free parking on premises", "Smoke alarm"]', '["Conditioner", "Room-darkening shades", "Hot water", "Self check-in", "Hair dryer", "Essentials", "Shampoo", "Bed linens", "Shower gel", "Building staff", "Iron", "Smoke alarm"]', '["Luggage dropoff allowed", "Conditioner", "Hot water", "Room-darkening shades", "Elevator", "Dedicated workspace", "Self check-in", "Hair dryer", "Essentials", "Air conditioning", "Shampoo", "Heating", "Bed linens", "TV", "Wifi", "Shower gel", "Building staff"]']


In [75]:
# Cell 103 — Encode amenities

import ast

amenity_values = set()

for value in df_paris['amenities']:
    try:
        items = ast.literal_eval(value)
        if isinstance(items, list):
            amenity_values.update(items)
    except:
        pass

amenity_values = sorted(amenity_values)

amenity_features = np.zeros(
    (len(df_paris), len(amenity_values))
)

for i, value in enumerate(df_paris['amenities']):
    try:
        items = set(ast.literal_eval(value))
        for j, amenity in enumerate(amenity_values):
            if amenity in items:
                amenity_features[i, j] = 1
    except:
        pass

print("Amenity features:", amenity_features.shape)
print("Number of unique amenities:", len(amenity_values))

Amenity features: (30971, 5187)
Number of unique amenities: 5187


In [76]:
# Cell 104 — Convert host response rate to numeric

df_paris['host_response_rate'] = (
    df_paris['host_response_rate']
    .astype(str)
    .str.replace('%', '', regex=False)
)

df_paris['host_response_rate'] = pd.to_numeric(
    df_paris['host_response_rate'],
    errors='coerce'
)

# Fill missing response rates with the mean
df_paris['host_response_rate'] = df_paris['host_response_rate'].fillna(
    df_paris['host_response_rate'].mean()
)

host_response_feature = df_paris['host_response_rate'].to_numpy().reshape(-1, 1)

print("Host response rate feature shape:", host_response_feature.shape)
print("Missing values:", np.isnan(host_response_feature).sum())

Host response rate feature shape: (30971, 1)
Missing values: 30971


In [77]:
# Cell 105 — Handle empty host response rate

df_paris = df_paris.drop(columns=['host_response_rate'])

# Since this feature cannot be used, remove it from our feature list
print("Host response rate removed.")
print("Remaining missing values:",
      df_paris.isna().sum().sum())

Host response rate removed.
Remaining missing values: 168866


In [78]:
# Cell 106 — Verify all feature blocks

continuous_features = df_paris[continuous_cols].to_numpy()

date_features = df_paris[date_cols].to_numpy()

print("Continuous:", continuous_features.shape)
print("Dates:", date_features.shape)
print("Categorical:", categorical_features.shape)
print("Text:", text_features.shape)
print("Amenities:", amenity_features.shape)
print("Host verifications:", verification_features.shape)

print("\nMissing values in feature blocks:")
print("Continuous:", np.isnan(continuous_features).sum())
print("Dates:", np.isnan(date_features).sum())
print("Categorical:", np.isnan(categorical_features).sum())
print("Text:", np.isnan(text_features).sum())
print("Amenities:", np.isnan(amenity_features).sum())

Continuous: (30971, 23)
Dates: (30971, 2)
Categorical: (30971, 86)
Text: (30971, 150)
Amenities: (30971, 5187)
Host verifications: (30971, 0)

Missing values in feature blocks:
Continuous: 0
Dates: 0
Categorical: 0
Text: 0
Amenities: 0


In [79]:
# Cell 107 — Assemble all Paris features

feature_blocks = [
    continuous_features,
    date_features,
    categorical_features,
    text_features,
    amenity_features
]

paris_features = np.concatenate(feature_blocks, axis=1)

print("Final Paris feature matrix:", paris_features.shape)

expected_features = 23 + 2 + 86 + 150 + 5187
print("Expected number of features:", expected_features)

Final Paris feature matrix: (30971, 5448)
Expected number of features: 5448


In [80]:
# Cell 108 — Create Paris target variable

paris_labels = np.log(df_paris['price'].to_numpy())

print("Target shape:", paris_labels.shape)
print("Log-price range:", paris_labels.min(), "to", paris_labels.max())

Target shape: (30971,)
Log-price range: 2.446685436967803 to 6.214608098422191


In [81]:
# Cell 109 — Shuffle and split Paris data

from sklearn.utils import shuffle

paris_features, paris_labels = shuffle(
    paris_features,
    paris_labels,
    random_state=42
)

num_data = len(paris_labels)

train_num = int(num_data * 0.7)
dev_num = int(num_data * 0.9)

X_train_paris = paris_features[:train_num]
X_dev_paris = paris_features[train_num:dev_num]
X_test_paris = paris_features[dev_num:]

y_train_paris = paris_labels[:train_num]
y_dev_paris = paris_labels[train_num:dev_num]
y_test_paris = paris_labels[dev_num:]

print("Train:", X_train_paris.shape, y_train_paris.shape)
print("Dev:", X_dev_paris.shape, y_dev_paris.shape)
print("Test:", X_test_paris.shape, y_test_paris.shape)

Train: (21679, 5448) (21679,)
Dev: (6194, 5448) (6194,)
Test: (3098, 5448) (3098,)


In [82]:
# Cell 110 — Normalize Paris features

mean = np.mean(X_train_paris, axis=0)
std = np.std(X_train_paris, axis=0)

# Prevent division by zero
std[std == 0] = 1

X_train_paris = (X_train_paris - mean) / std
X_dev_paris = (X_dev_paris - mean) / std
X_test_paris = (X_test_paris - mean) / std

print("Normalization complete!")
print("Train mean:", np.mean(X_train_paris))
print("Train std:", np.std(X_train_paris))

Normalization complete!
Train mean: 3.3314705214867302e-15
Train std: 0.8952066841475039


In [83]:
# Cell 111 — Paris Linear Regression

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

linear_model_paris = LinearRegression()

linear_model_paris.fit(X_train_paris, y_train_paris)

linear_dev_pred = linear_model_paris.predict(X_dev_paris)

linear_dev_mse = mean_squared_error(y_dev_paris, linear_dev_pred)
linear_dev_r2 = r2_score(y_dev_paris, linear_dev_pred)

print("Linear Regression — Paris")
print("Dev MSE:", linear_dev_mse)
print("Dev R²:", linear_dev_r2)

Linear Regression — Paris
Dev MSE: 0.09080119853669824
Dev R²: 0.5909986802011447


In [84]:
# Cell 112 — Paris KNN Regression

from sklearn.neighbors import KNeighborsRegressor

knn_model_paris = KNeighborsRegressor(n_neighbors=5)

knn_model_paris.fit(X_train_paris, y_train_paris)

knn_dev_pred = knn_model_paris.predict(X_dev_paris)

knn_dev_mse = mean_squared_error(y_dev_paris, knn_dev_pred)
knn_dev_r2 = r2_score(y_dev_paris, knn_dev_pred)

print("KNN Regression — Paris")
print("Dev MSE:", knn_dev_mse)
print("Dev R²:", knn_dev_r2)

KNN Regression — Paris
Dev MSE: 0.15769230020989605
Dev R²: 0.2896970530416739


In [85]:
# Cell 113 — Paris Random Forest

from sklearn.ensemble import RandomForestRegressor

rf_model_paris = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model_paris.fit(X_train_paris, y_train_paris)

rf_dev_pred = rf_model_paris.predict(X_dev_paris)

rf_dev_mse = mean_squared_error(y_dev_paris, rf_dev_pred)
rf_dev_r2 = r2_score(y_dev_paris, rf_dev_pred)

print("Random Forest — Paris")
print("Dev MSE:", rf_dev_mse)
print("Dev R²:", rf_dev_r2)

Random Forest — Paris
Dev MSE: 0.0806896994379312
Dev R²: 0.6365445159741077


In [86]:
# Cell 114 — Paris XGBoost

from xgboost import XGBRegressor

xgb_model_paris = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

xgb_model_paris.fit(X_train_paris, y_train_paris)

xgb_dev_pred = xgb_model_paris.predict(X_dev_paris)

xgb_dev_mse = mean_squared_error(y_dev_paris, xgb_dev_pred)
xgb_dev_r2 = r2_score(y_dev_paris, xgb_dev_pred)

print("XGBoost — Paris")
print("Dev MSE:", xgb_dev_mse)
print("Dev R²:", xgb_dev_r2)

XGBoost — Paris
Dev MSE: 0.0667837456441316
Dev R²: 0.6991819430828174


In [87]:
# Cell 115 — Paris Neural Network

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# Convert NumPy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train_paris, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_paris, dtype=torch.float32).view(-1, 1)

X_dev_tensor = torch.tensor(X_dev_paris, dtype=torch.float32)
y_dev_tensor = torch.tensor(y_dev_paris, dtype=torch.float32).view(-1, 1)

# Neural network architecture from the report
nn_model_paris = nn.Sequential(
    nn.Linear(X_train_paris.shape[1], 128),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(128, 512),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(512, 64),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(64, 1)
)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    nn_model_paris.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

best_dev_loss = float("inf")
best_epoch = 0

for epoch in range(100):

    nn_model_paris.train()

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()

        predictions = nn_model_paris(X_batch)
        loss = criterion(predictions, y_batch)

        loss.backward()
        optimizer.step()

    # Validation
    nn_model_paris.eval()

    with torch.no_grad():
        dev_predictions = nn_model_paris(X_dev_tensor)
        dev_loss = criterion(dev_predictions, y_dev_tensor).item()

    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        best_epoch = epoch + 1
        best_model_state_paris = nn_model_paris.state_dict()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/100 — Dev Loss: {dev_loss:.6f}")

# Restore best model
nn_model_paris.load_state_dict(best_model_state_paris)

print("\nParis Neural Network finished!")
print("Best epoch:", best_epoch)
print("Best Dev MSE:", best_dev_loss)

Epoch 10/100 — Dev Loss: 0.493616
Epoch 20/100 — Dev Loss: 0.265202
Epoch 30/100 — Dev Loss: 0.264392
Epoch 40/100 — Dev Loss: 0.157662
Epoch 50/100 — Dev Loss: 0.162511
Epoch 60/100 — Dev Loss: 0.133928
Epoch 70/100 — Dev Loss: 0.123520
Epoch 80/100 — Dev Loss: 0.115490
Epoch 90/100 — Dev Loss: 0.127891
Epoch 100/100 — Dev Loss: 0.121583

Paris Neural Network finished!
Best epoch: 94
Best Dev MSE: 0.10062310844659805


In [88]:
# Cell 116 — Paris Linear Regression Test

linear_test_pred_paris = linear_model_paris.predict(X_test_paris)

linear_test_mse_paris = mean_squared_error(
    y_test_paris,
    linear_test_pred_paris
)

linear_test_r2_paris = r2_score(
    y_test_paris,
    linear_test_pred_paris
)

print("Linear Regression — Paris Test")
print("Test MSE:", linear_test_mse_paris)
print("Test R²:", linear_test_r2_paris)

Linear Regression — Paris Test
Test MSE: 0.08857046212318814
Test R²: 0.6200668230365037


In [89]:
# Cell 117 — Paris KNN Test

knn_test_pred_paris = knn_model_paris.predict(X_test_paris)

knn_test_mse_paris = mean_squared_error(
    y_test_paris,
    knn_test_pred_paris
)

knn_test_r2_paris = r2_score(
    y_test_paris,
    knn_test_pred_paris
)

print("KNN — Paris Test")
print("Test MSE:", knn_test_mse_paris)
print("Test R²:", knn_test_r2_paris)

KNN — Paris Test
Test MSE: 0.16016840246891983
Test R²: 0.3129392289435402


In [90]:
# Cell 118 — Paris Random Forest Test

rf_test_pred_paris = rf_model_paris.predict(X_test_paris)

rf_test_mse_paris = mean_squared_error(
    y_test_paris,
    rf_test_pred_paris
)

rf_test_r2_paris = r2_score(
    y_test_paris,
    rf_test_pred_paris
)

print("Random Forest — Paris Test")
print("Test MSE:", rf_test_mse_paris)
print("Test R²:", rf_test_r2_paris)

Random Forest — Paris Test
Test MSE: 0.0822615794694284
Test R²: 0.6471295002798383


In [91]:
# Cell 119 — Paris XGBoost Test

xgb_test_pred_paris = xgb_model_paris.predict(X_test_paris)

xgb_test_mse_paris = mean_squared_error(
    y_test_paris,
    xgb_test_pred_paris
)

xgb_test_r2_paris = r2_score(
    y_test_paris,
    xgb_test_pred_paris
)

print("XGBoost — Paris Test")
print("Test MSE:", xgb_test_mse_paris)
print("Test R²:", xgb_test_r2_paris)

XGBoost — Paris Test
Test MSE: 0.06465327196934958
Test R²: 0.7226623590804433


In [92]:
# Cell 120 — Paris Neural Network Test

nn_model_paris.eval()

with torch.no_grad():
    nn_test_pred_paris = nn_model_paris(X_test_tensor if 'X_test_tensor' in globals() else torch.tensor(X_test_paris, dtype=torch.float32))

nn_test_pred_paris = nn_test_pred_paris.numpy().flatten()

nn_test_mse_paris = mean_squared_error(
    y_test_paris,
    nn_test_pred_paris
)

nn_test_r2_paris = r2_score(
    y_test_paris,
    nn_test_pred_paris
)

print("Neural Network — Paris Test")
print("Test MSE:", nn_test_mse_paris)
print("Test R²:", nn_test_r2_paris)

Neural Network — Paris Test
Test MSE: 0.12151235239500073
Test R²: 0.4787587985989077


## NYC baselines


In [93]:
# Cell 127 — Load NYC Detailed Listings

df_nyc = pd.read_csv("/content/gdrive/MyDrive/airbnb_project/data/raw/listings_nyc26.csv")

print("NYC dataset shape:", df_nyc.shape)
print("\nPrice column:")
print(df_nyc["price"].head())
print("\nNon-null prices:", df_nyc["price"].notna().sum())

NYC dataset shape: (30259, 90)

Price column:
0    $113.97
1    $117.27
2     $80.06
3     $77.17
4    $202.47
Name: price, dtype: object

Non-null prices: 21515


In [94]:
# Cell 128 — Clean NYC Price

df_nyc["price"] = (
    df_nyc["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

df_nyc = df_nyc.dropna(subset=["price"]).copy()

print("After removing missing prices:", df_nyc.shape)
print("Price range:", df_nyc["price"].min(), "to", df_nyc["price"].max())

After removing missing prices: (21515, 90)
Price range: 4.58 to 30972.96


In [95]:
# Cell 129 — Remove extreme price outliers

df_nyc = df_nyc[df_nyc["price"] <= 500].copy()

print("After price <= $500:", df_nyc.shape)
print("Price range:", df_nyc["price"].min(), "to", df_nyc["price"].max())

After price <= $500: (19148, 90)
Price range: 4.58 to 500.0


In [96]:
# Cell 130 — Drop metadata columns

metadata_cols = [
    "id",
    "listing_url",
    "scrape_id",
    "last_scraped",
    "source",
    "picture_url",
    "host_url",
    "host_thumbnail_url",
    "host_picture_url"
]

df_nyc = df_nyc.drop(
    columns=[col for col in metadata_cols if col in df_nyc.columns]
)

print("NYC shape after dropping metadata:", df_nyc.shape)

NYC shape after dropping metadata: (19148, 81)


In [97]:
# Cell 131 — Inspect NYC feature columns

print("Columns:")
print(df_nyc.columns.tolist())

print("\nMissing values in important columns:")
check_cols = [
    "host_response_time",
    "host_response_rate",
    "host_is_superhost",
    "host_verifications",
    "amenities",
    "host_since",
    "first_review",
    "last_review",
    "name",
    "description",
    "neighborhood_overview",
    "host_about"
]

for col in check_cols:
    if col in df_nyc.columns:
        print(f"{col}: {df_nyc[col].isna().sum()} missing")
    else:
        print(f"{col}: NOT PRESENT")

Columns:
['name', 'description', 'neighborhood_overview', 'host_id', 'host_profile_id', 'host_profile_url', 'host_name', 'host_since', 'hosts_time_as_user_years', 'hosts_time_as_user_months', 'hosts_time_as_host_years', 'hosts_time_as_host_months', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price', 'price_quote_price_per_night', 'price_quote_raw', 'minimum_nights', 'maximum_nights', 'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights', 'maximum_maximum_n

In [98]:
# Cell 132 — Remove completely empty columns

empty_cols = df_nyc.columns[df_nyc.isna().all()].tolist()

print("Completely empty columns:")
print(empty_cols)

df_nyc = df_nyc.drop(columns=empty_cols)

print("\nNYC shape after removing empty columns:", df_nyc.shape)

Completely empty columns:
['neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'calendar_updated', 'instant_bookable']

NYC shape after removing empty columns: (19148, 70)


In [99]:
# Cell 133 — Inspect numeric columns and missing values

numeric_cols_nyc = df_nyc.select_dtypes(include=np.number).columns.tolist()

print("Number of numeric columns:", len(numeric_cols_nyc))
print("\nNumeric columns with missing values:")

for col in numeric_cols_nyc:
    missing = df_nyc[col].isna().sum()
    if missing > 0:
        print(f"{col}: {missing} missing")

Number of numeric columns: 47

Numeric columns with missing values:
host_profile_id: 245 missing
hosts_time_as_user_years: 245 missing
hosts_time_as_user_months: 245 missing
hosts_time_as_host_years: 245 missing
hosts_time_as_host_months: 245 missing
host_listings_count: 245 missing
bathrooms: 2105 missing
bedrooms: 7342 missing
beds: 1580 missing
price_quote_total_price: 1 missing
price_quote_price_per_night: 1 missing
minimum_nights: 1 missing
maximum_nights: 1 missing
minimum_minimum_nights: 1 missing
maximum_minimum_nights: 1 missing
minimum_maximum_nights: 1 missing
maximum_maximum_nights: 1 missing
review_scores_rating: 5055 missing
review_scores_accuracy: 5055 missing
review_scores_cleanliness: 5055 missing
review_scores_checkin: 5055 missing
review_scores_communication: 5055 missing
review_scores_location: 5056 missing
review_scores_value: 5055 missing
reviews_per_month: 5055 missing


In [100]:
# Cell 134 — Define NYC continuous features

continuous_cols_nyc = [
    "host_listings_count",
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "maximum_nights",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "calculated_host_listings_count",
    "reviews_per_month"
]

print("Number of continuous features:", len(continuous_cols_nyc))

print("\nMissing values:")
for col in continuous_cols_nyc:
    print(f"{col}: {df_nyc[col].isna().sum()}")

Number of continuous features: 25

Missing values:
host_listings_count: 245
latitude: 0
longitude: 0
accommodates: 0
bathrooms: 2105
bedrooms: 7342
beds: 1580
minimum_nights: 1
maximum_nights: 1
availability_30: 0
availability_60: 0
availability_90: 0
availability_365: 0
number_of_reviews: 0
number_of_reviews_ltm: 0
number_of_reviews_l30d: 0
review_scores_rating: 5055
review_scores_accuracy: 5055
review_scores_cleanliness: 5055
review_scores_checkin: 5055
review_scores_communication: 5055
review_scores_location: 5056
review_scores_value: 5055
calculated_host_listings_count: 0
reviews_per_month: 5055


In [101]:
# Cell 135 — Fill missing continuous values with median

for col in continuous_cols_nyc:
    median_value = df_nyc[col].median()
    df_nyc[col] = df_nyc[col].fillna(median_value)

print("Remaining missing values in continuous features:")

for col in continuous_cols_nyc:
    print(f"{col}: {df_nyc[col].isna().sum()}")

Remaining missing values in continuous features:
host_listings_count: 0
latitude: 0
longitude: 0
accommodates: 0
bathrooms: 0
bedrooms: 0
beds: 0
minimum_nights: 0
maximum_nights: 0
availability_30: 0
availability_60: 0
availability_90: 0
availability_365: 0
number_of_reviews: 0
number_of_reviews_ltm: 0
number_of_reviews_l30d: 0
review_scores_rating: 0
review_scores_accuracy: 0
review_scores_cleanliness: 0
review_scores_checkin: 0
review_scores_communication: 0
review_scores_location: 0
review_scores_value: 0
calculated_host_listings_count: 0
reviews_per_month: 0


In [102]:
# Cell 136 — Define NYC categorical features

categorical_cols_nyc = [
    "host_is_superhost",
    "host_has_profile_pic",
    "host_identity_verified",
    "neighbourhood_cleansed",
    "neighbourhood_group_cleansed",
    "property_type",
    "room_type",
    "has_availability",
    "license"
]

# Keep only columns that actually exist
categorical_cols_nyc = [
    col for col in categorical_cols_nyc
    if col in df_nyc.columns
]

print("Categorical features:")
print(categorical_cols_nyc)

print("\nMissing values:")
for col in categorical_cols_nyc:
    print(f"{col}: {df_nyc[col].isna().sum()}")

Categorical features:
['host_is_superhost', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'property_type', 'room_type', 'has_availability', 'license']

Missing values:
host_is_superhost: 245
host_has_profile_pic: 245
host_identity_verified: 245
neighbourhood_cleansed: 0
neighbourhood_group_cleansed: 0
property_type: 0
room_type: 0
has_availability: 0
license: 15538


In [103]:
# Cell 137 — Fill missing categorical values

for col in categorical_cols_nyc:
    df_nyc[col] = df_nyc[col].fillna("Unknown").astype(str)

print("Remaining missing values:")

for col in categorical_cols_nyc:
    print(f"{col}: {df_nyc[col].isna().sum()}")

Remaining missing values:
host_is_superhost: 0
host_has_profile_pic: 0
host_identity_verified: 0
neighbourhood_cleansed: 0
neighbourhood_group_cleansed: 0
property_type: 0
room_type: 0
has_availability: 0
license: 0


In [104]:
# Cell 138 — Inspect NYC date columns

date_cols_nyc = [
    "first_review",
    "last_review"
]

print("Date columns:")
print(date_cols_nyc)

print("\nMissing values:")
for col in date_cols_nyc:
    print(f"{col}: {df_nyc[col].isna().sum()} missing")

print("\nSample values:")
for col in date_cols_nyc:
    print(f"\n{col}:")
    print(df_nyc[col].dropna().head())

Date columns:
['first_review', 'last_review']

Missing values:
first_review: 5055 missing
last_review: 5055 missing

Sample values:

first_review:
0    2015-12-04
1    2009-05-25
2    2022-06-05
3    2009-10-28
4    2010-01-16
Name: first_review, dtype: object

last_review:
0    2026-05-28
1    2026-04-23
2    2025-10-07
3    2026-01-11
4    2025-09-23
Name: last_review, dtype: object


In [105]:
# Cell 139 — Convert date columns to numeric features

date_features_nyc = []

for col in date_cols_nyc:
    dates = pd.to_datetime(df_nyc[col], errors="coerce")

    min_date = dates.min()
    mean_date = dates.mean()

    # Convert to days since earliest date
    numeric_dates = (dates - min_date).dt.total_seconds() / (24 * 60 * 60)

    # Replace missing dates with mean
    numeric_dates = numeric_dates.fillna(
        numeric_dates.mean()
    )

    date_features_nyc.append(numeric_dates.to_numpy())

date_features_nyc = np.column_stack(date_features_nyc)

print("Date feature shape:", date_features_nyc.shape)
print("Missing values:", np.isnan(date_features_nyc).sum())
print("First few rows:")
print(date_features_nyc[:5])

Date feature shape: (19148, 2)
Missing values: 0
First few rows:
[[2384. 5495.]
 [   0. 5460.]
 [4759. 5262.]
 [ 156. 5358.]
 [ 236. 5248.]]


In [106]:
# Cell 140 — Inspect NYC text columns

text_cols_nyc = [
    "name",
    "description",
    "host_about"
]

print("Text columns:")
for col in text_cols_nyc:
    print(f"{col}: {df_nyc[col].isna().sum()} missing")

Text columns:
name: 0 missing
description: 474 missing
host_about: 7977 missing


In [107]:
# Cell 141 — Prepare NYC text data

for col in text_cols_nyc:
    df_nyc[col] = df_nyc[col].fillna("").astype(str)

print("Remaining missing values:")

for col in text_cols_nyc:
    print(f"{col}: {df_nyc[col].isna().sum()}")

Remaining missing values:
name: 0
description: 0
host_about: 0


In [108]:
# Cell 142 — TF-IDF + SVD for NYC text

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

text_features_nyc = []

for col in text_cols_nyc:
    print(f"Processing: {col}")

    vectorizer = TfidfVectorizer(
        max_features=5000,
        stop_words="english"
    )

    tfidf = vectorizer.fit_transform(df_nyc[col])

    svd = TruncatedSVD(
        n_components=50,
        random_state=42
    )

    reduced = svd.fit_transform(tfidf)

    text_features_nyc.append(reduced)

    print("  TF-IDF shape:", tfidf.shape)
    print("  SVD shape:", reduced.shape)

text_features_nyc = np.hstack(text_features_nyc)

print("\nFinal text feature shape:", text_features_nyc.shape)
print("Missing values:", np.isnan(text_features_nyc).sum())

Processing: name
  TF-IDF shape: (19148, 5000)
  SVD shape: (19148, 50)
Processing: description
  TF-IDF shape: (19148, 5000)
  SVD shape: (19148, 50)
Processing: host_about
  TF-IDF shape: (19148, 5000)
  SVD shape: (19148, 50)

Final text feature shape: (19148, 150)
Missing values: 0


In [109]:
# Cell 143 — One-hot encode NYC categorical features

from sklearn.preprocessing import OneHotEncoder

encoder_nyc = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

categorical_features_nyc = encoder_nyc.fit_transform(
    df_nyc[categorical_cols_nyc]
)

print("Categorical feature shape:", categorical_features_nyc.shape)
print("Missing values:", np.isnan(categorical_features_nyc).sum())

Categorical feature shape: (19148, 2010)
Missing values: 0


In [110]:
# Cell 144 — Inspect NYC amenities

print("Number of listings:", len(df_nyc))

print("\nFirst 3 amenities values:\n")
for i, value in enumerate(df_nyc["amenities"].head(3)):
    print(f"{i}: {value}")

Number of listings: 19148

First 3 amenities values:

0: ["Bed linens", "Hot water", "Exterior security cameras on property", "Cleaning products", "Fire extinguisher", "Elevator", "Lockbox", "Microwave", "Laundromat nearby", "Oven", "Portable fans", "Self check-in", "Paid parking off premises", "Kitchen", "Free street parking", "Dining table", "Long term stays allowed", "Hair dryer", "Body soap", "Extra pillows and blankets", "Washer", "Backyard", "Dryer", "Iron", "Bikes", "Ceiling fan", "Carbon monoxide alarm", "Dishes and silverware", "Freezer", "Stove", "Heating", "Coffee maker", "Luggage dropoff allowed", "Conditioner", "Bathtub", "Wine glasses", "Clothing storage", "Fast wifi \u2013 330 Mbps", "Cleaning available during stay", "Blender", "Shower gel", "Smoke alarm", "Cooking basics", "Refrigerator", "Hangers", "Single level home", "Dedicated workspace", "Essentials", "Shampoo"]
1: ["Bed linens", "Hot water", "Fire extinguisher", "Dishwasher", "Microwave", "Oven", "Kitchen", "Free 

In [111]:
# Cell 145 — Parse NYC amenities

import ast

def parse_amenities_nyc(value):
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(x).strip() for x in parsed]
        return []
    except:
        return []

amenities_parsed_nyc = df_nyc["amenities"].apply(parse_amenities_nyc)

print("First 3 parsed listings:\n")
for i, value in enumerate(amenities_parsed_nyc.head(3)):
    print(f"{i}: {value}")

print("\nNumber of listings:", len(amenities_parsed_nyc))

First 3 parsed listings:

0: ['Bed linens', 'Hot water', 'Exterior security cameras on property', 'Cleaning products', 'Fire extinguisher', 'Elevator', 'Lockbox', 'Microwave', 'Laundromat nearby', 'Oven', 'Portable fans', 'Self check-in', 'Paid parking off premises', 'Kitchen', 'Free street parking', 'Dining table', 'Long term stays allowed', 'Hair dryer', 'Body soap', 'Extra pillows and blankets', 'Washer', 'Backyard', 'Dryer', 'Iron', 'Bikes', 'Ceiling fan', 'Carbon monoxide alarm', 'Dishes and silverware', 'Freezer', 'Stove', 'Heating', 'Coffee maker', 'Luggage dropoff allowed', 'Conditioner', 'Bathtub', 'Wine glasses', 'Clothing storage', 'Fast wifi – 330 Mbps', 'Cleaning available during stay', 'Blender', 'Shower gel', 'Smoke alarm', 'Cooking basics', 'Refrigerator', 'Hangers', 'Single level home', 'Dedicated workspace', 'Essentials', 'Shampoo']
1: ['Bed linens', 'Hot water', 'Fire extinguisher', 'Dishwasher', 'Microwave', 'Oven', 'Kitchen', 'Free street parking', 'Washer', 'Hair 

In [112]:
# Cell 146 — Create binary amenity features

all_amenities_nyc = sorted(
    set(
        amenity
        for listing in amenities_parsed_nyc
        for amenity in listing
    )
)

amenity_to_index_nyc = {
    amenity: i
    for i, amenity in enumerate(all_amenities_nyc)
}

amenities_features_nyc = np.zeros(
    (len(df_nyc), len(all_amenities_nyc)),
    dtype=np.float32
)

for i, listing in enumerate(amenities_parsed_nyc):
    for amenity in listing:
        if amenity in amenity_to_index_nyc:
            amenities_features_nyc[i, amenity_to_index_nyc[amenity]] = 1

print("Number of unique amenities:", len(all_amenities_nyc))
print("Amenity feature shape:", amenities_features_nyc.shape)
print("Missing values:", np.isnan(amenities_features_nyc).sum())

Number of unique amenities: 4995
Amenity feature shape: (19148, 4995)
Missing values: 0


In [113]:
# Cell 147 — Combine all NYC feature blocks

continuous_features_nyc = df_nyc[continuous_cols_nyc].to_numpy()

X_nyc = np.hstack([
    continuous_features_nyc,
    date_features_nyc,
    categorical_features_nyc,
    text_features_nyc,
    amenities_features_nyc
])

print("Final NYC feature matrix shape:", X_nyc.shape)
print("Missing values:", np.isnan(X_nyc).sum())

Final NYC feature matrix shape: (19148, 7182)
Missing values: 0


In [114]:
# Cell 148 — Prepare NYC target variable

y_nyc = np.log(df_nyc["price"].to_numpy())

print("Target shape:", y_nyc.shape)
print("Minimum log(price):", y_nyc.min())
print("Maximum log(price):", y_nyc.max())
print("Missing values:", np.isnan(y_nyc).sum())

Target shape: (19148,)
Minimum log(price): 1.5216989981260935
Maximum log(price): 6.214608098422191
Missing values: 0


In [115]:
# Cell 149 — Shuffle and split NYC data

from sklearn.utils import shuffle

X_nyc, y_nyc = shuffle(
    X_nyc,
    y_nyc,
    random_state=42
)

num_data_nyc = len(y_nyc)

train_num_nyc = int(num_data_nyc * 0.7)
dev_num_nyc = int(num_data_nyc * 0.9)

X_train_nyc = X_nyc[:train_num_nyc]
y_train_nyc = y_nyc[:train_num_nyc]

X_dev_nyc = X_nyc[train_num_nyc:dev_num_nyc]
y_dev_nyc = y_nyc[train_num_nyc:dev_num_nyc]

X_test_nyc = X_nyc[dev_num_nyc:]
y_test_nyc = y_nyc[dev_num_nyc:]

print("Train:", X_train_nyc.shape, y_train_nyc.shape)
print("Dev:", X_dev_nyc.shape, y_dev_nyc.shape)
print("Test:", X_test_nyc.shape, y_test_nyc.shape)

Train: (13403, 7182) (13403,)
Dev: (3830, 7182) (3830,)
Test: (1915, 7182) (1915,)


In [116]:
# Cell 150 — Normalize NYC features

train_mean_nyc = X_train_nyc.mean(axis=0)
train_std_nyc = X_train_nyc.std(axis=0)

# Prevent division by zero for constant features
train_std_nyc[train_std_nyc == 0] = 1

X_train_nyc = (X_train_nyc - train_mean_nyc) / train_std_nyc
X_dev_nyc = (X_dev_nyc - train_mean_nyc) / train_std_nyc
X_test_nyc = (X_test_nyc - train_mean_nyc) / train_std_nyc

print("Train shape:", X_train_nyc.shape)
print("Dev shape:", X_dev_nyc.shape)
print("Test shape:", X_test_nyc.shape)

print("\nNaN values:")
print("Train:", np.isnan(X_train_nyc).sum())
print("Dev:", np.isnan(X_dev_nyc).sum())
print("Test:", np.isnan(X_test_nyc).sum())

Train shape: (13403, 7182)
Dev shape: (3830, 7182)
Test shape: (1915, 7182)

NaN values:
Train: 0
Dev: 0
Test: 0


In [117]:
# Cell 151 — NYC Linear Regression baseline

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

linear_nyc = LinearRegression()

linear_nyc.fit(X_train_nyc, y_train_nyc)

y_dev_pred_linear_nyc = linear_nyc.predict(X_dev_nyc)

linear_dev_mse_nyc = mean_squared_error(
    y_dev_nyc,
    y_dev_pred_linear_nyc
)

linear_dev_r2_nyc = r2_score(
    y_dev_nyc,
    y_dev_pred_linear_nyc
)

print("NYC Linear Regression — Dev")
print("MSE:", linear_dev_mse_nyc)
print("R²:", linear_dev_r2_nyc)

NYC Linear Regression — Dev
MSE: 0.12701081598930508
R²: 0.7050569231008146


In [118]:
# Cell 152 — NYC KNN Regression baseline

from sklearn.neighbors import KNeighborsRegressor

knn_nyc = KNeighborsRegressor(n_neighbors=5)

knn_nyc.fit(X_train_nyc, y_train_nyc)

y_dev_pred_knn_nyc = knn_nyc.predict(X_dev_nyc)

knn_dev_mse_nyc = mean_squared_error(
    y_dev_nyc,
    y_dev_pred_knn_nyc
)

knn_dev_r2_nyc = r2_score(
    y_dev_nyc,
    y_dev_pred_knn_nyc
)

print("NYC KNN Regression — Dev")
print("MSE:", knn_dev_mse_nyc)
print("R²:", knn_dev_r2_nyc)

NYC KNN Regression — Dev
MSE: 0.2547652548665277
R²: 0.408387013562167


In [119]:
# Cell 153 — NYC Random Forest baseline

from sklearn.ensemble import RandomForestRegressor

rf_nyc = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_nyc.fit(X_train_nyc, y_train_nyc)

y_dev_pred_rf_nyc = rf_nyc.predict(X_dev_nyc)

rf_dev_mse_nyc = mean_squared_error(
    y_dev_nyc,
    y_dev_pred_rf_nyc
)

rf_dev_r2_nyc = r2_score(
    y_dev_nyc,
    y_dev_pred_rf_nyc
)

print("NYC Random Forest — Dev")
print("MSE:", rf_dev_mse_nyc)
print("R²:", rf_dev_r2_nyc)

NYC Random Forest — Dev
MSE: 0.08497619672667525
R²: 0.8026692393829946


In [120]:
# Cell 154 — NYC XGBoost baseline

from xgboost import XGBRegressor

xgb_nyc = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

xgb_nyc.fit(X_train_nyc, y_train_nyc)

y_dev_pred_xgb_nyc = xgb_nyc.predict(X_dev_nyc)

xgb_dev_mse_nyc = mean_squared_error(
    y_dev_nyc,
    y_dev_pred_xgb_nyc
)

xgb_dev_r2_nyc = r2_score(
    y_dev_nyc,
    y_dev_pred_xgb_nyc
)

print("NYC XGBoost — Dev")
print("MSE:", xgb_dev_mse_nyc)
print("R²:", xgb_dev_r2_nyc)

NYC XGBoost — Dev
MSE: 0.07842638268431817
R²: 0.8178791432933273


In [121]:
# Cell 155 — Define NYC Neural Network

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

nn_nyc = Sequential([
    Dense(128, activation="relu", input_shape=(X_train_nyc.shape[1],)),
    Dropout(0.2),
    Dense(512, activation="relu"),
    Dropout(0.2),
    Dense(64, activation="relu"),
    Dense(1)
])

nn_nyc.compile(
    optimizer=Adam(
        learning_rate=1e-4,
        weight_decay=1e-5
    ),
    loss="mse"
)

nn_nyc.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       919,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 512)            │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        32,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,018,369 (3.88 MB)

 Trainable params: 1,018,369 (3.88 MB)

 Non-trainable params: 0 (0.00 B)

In [122]:
# Cell 156 — Train NYC Neural Network

from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint_path_nyc = "/content/best_nyc_nn.keras"

checkpoint_nyc = ModelCheckpoint(
    checkpoint_path_nyc,
    monitor="val_loss",
    save_best_only=True,
    mode="min"
)

history_nyc = nn_nyc.fit(
    X_train_nyc,
    y_train_nyc,
    validation_data=(X_dev_nyc, y_dev_nyc),
    epochs=100,
    batch_size=64,
    callbacks=[checkpoint_nyc],
    verbose=1
)

print("\nBest validation loss:", min(history_nyc.history["val_loss"]))
print(
    "Best epoch:",
    np.argmin(history_nyc.history["val_loss"]) + 1
)

Epoch 1/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - loss: 4.9805 - val_loss: 0.6244
Epoch 2/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.9888 - val_loss: 0.6033
Epoch 3/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.5855 - val_loss: 0.5198
Epoch 4/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.4330 - val_loss: 0.5331
Epoch 5/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3652 - val_loss: 0.5230
Epoch 6/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3092 - val_loss: 0.5515
Epoch 7/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2936 - val_loss: 0.4825
Epoch 8/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2661 - val_loss: 0.5394
Epoch 9/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.2498 - val_loss: 0.5529
Epoch 10/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2262 - val_loss: 0.5169
Epoch 11/100
210/210 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.2187 - val_loss: 0.5776
Epoch 12/100
210/210 ━━━━━━━━━━━━━━━━━━

In [123]:
# Cell 157 — Evaluate best NYC Neural Network on test set

from tensorflow.keras.models import load_model

best_nn_nyc = load_model(checkpoint_path_nyc)

y_test_pred_nn_nyc = best_nn_nyc.predict(
    X_test_nyc,
    verbose=0
).ravel()

nn_test_mse_nyc = mean_squared_error(
    y_test_nyc,
    y_test_pred_nn_nyc
)

nn_test_r2_nyc = r2_score(
    y_test_nyc,
    y_test_pred_nn_nyc
)

print("NYC Neural Network — Test")
print("MSE:", nn_test_mse_nyc)
print("R²:", nn_test_r2_nyc)

NYC Neural Network — Test
MSE: 0.466216115708226
R²: -0.09960783152836905


In [124]:
# Cell 158 — NYC Linear Regression — Test

y_test_pred_linear_nyc = linear_nyc.predict(X_test_nyc)

linear_test_mse_nyc = mean_squared_error(
    y_test_nyc,
    y_test_pred_linear_nyc
)

linear_test_r2_nyc = r2_score(
    y_test_nyc,
    y_test_pred_linear_nyc
)

print("NYC Linear Regression — Test")
print("MSE:", linear_test_mse_nyc)
print("R²:", linear_test_r2_nyc)

NYC Linear Regression — Test
MSE: 0.12181443520629484
R²: 0.71269095504653


In [125]:
# Cell 159 — NYC KNN Regression — Test

y_test_pred_knn_nyc = knn_nyc.predict(X_test_nyc)

knn_test_mse_nyc = mean_squared_error(
    y_test_nyc,
    y_test_pred_knn_nyc
)

knn_test_r2_nyc = r2_score(
    y_test_nyc,
    y_test_pred_knn_nyc
)

print("NYC KNN Regression — Test")
print("MSE:", knn_test_mse_nyc)
print("R²:", knn_test_r2_nyc)

NYC KNN Regression — Test
MSE: 0.24781091362638136
R²: 0.4155182281765959


In [126]:
# Cell 160 — NYC Random Forest — Test

y_test_pred_rf_nyc = rf_nyc.predict(X_test_nyc)

rf_test_mse_nyc = mean_squared_error(
    y_test_nyc,
    y_test_pred_rf_nyc
)

rf_test_r2_nyc = r2_score(
    y_test_nyc,
    y_test_pred_rf_nyc
)

print("NYC Random Forest — Test")
print("MSE:", rf_test_mse_nyc)
print("R²:", rf_test_r2_nyc)

NYC Random Forest — Test
MSE: 0.08512127111987755
R²: 0.7992347042510967


In [127]:
# Cell 161 — NYC XGBoost — Test

y_test_pred_xgb_nyc = xgb_nyc.predict(X_test_nyc)

xgb_test_mse_nyc = mean_squared_error(
    y_test_nyc,
    y_test_pred_xgb_nyc
)

xgb_test_r2_nyc = r2_score(
    y_test_nyc,
    y_test_pred_xgb_nyc
)

print("NYC XGBoost — Test")
print("MSE:", xgb_test_mse_nyc)
print("R²:", xgb_test_r2_nyc)

NYC XGBoost — Test
MSE: 0.07999335350838883
R²: 0.8113293062501278


## **Cross city Prediction**



In [128]:
from google.colab import drive
drive.mount('/content/gdrive')

import pandas as pd, numpy as np, ast, warnings
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import mean_squared_error, r2_score
warnings.simplefilter('ignore')
RAW = '/content/gdrive/MyDrive/airbnb_project/data/raw/'

def load(f):
    d = pd.read_csv(RAW + f, low_memory=False)
    d['price'] = pd.to_numeric(d['price'].astype(str).str.replace('[$,]', '', regex=True), errors='coerce')
    d = d.dropna(subset=['price'])
    return d[(d.price > 0) & (d.price <= 500)].reset_index(drop=True)

nyc = load('listings_nyc26.csv')
paris = load('listings_pari26.csv')
berlin = load('listings_detail_berlin.csv')
print(len(nyc), len(paris), len(berlin)) # expect 19148 30971 9133

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
19148 30971 9133


In [129]:
CONT = ['latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'maximum_nights',
        'availability_30', 'availability_60', 'availability_90', 'availability_365', 'number_of_reviews', 'number_of_reviews_ltm',
        'number_of_reviews_l30d', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
        'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value',
        'calculated_host_listings_count', 'host_listings_count', 'reviews_per_month']
CAT = ['room_type', 'property_type', 'host_is_superhost', 'host_has_profile_pic',
       'host_identity_verified', 'has_availability', 'instant_bookable']

def parse(v):
    try:
        x = ast.literal_eval(v)
        return [str(i) for i in x] if isinstance(x, list) else []
    except Exception:
        return []

def make_spec(train):
    s = {'med': train[CONT].median().fillna(0)}
    s['cats'] = {c: sorted(train[c].fillna('Unknown').astype(str).unique()) for c in CAT}
    cnt = pd.Series([i for v in train['amenities'] for i in set(parse(v))]).value_counts()
    s['amen'] = sorted(cnt[cnt >= 0.01 * len(train)].index)
    return s

def transform(df, s):
    parts = [df[CONT].fillna(s['med']).fillna(0).to_numpy(float)]
    for c in CAT:
        v = df[c].fillna('Unknown').astype(str)
        parts.append(np.column_stack([(v == k).to_numpy(float) for k in s['cats'][c]]))
    mlb = MultiLabelBinarizer(classes=s['amen'])
    mlb.fit([[]])
    parts.append(mlb.transform([parse(v) for v in df['amenities']]).astype(float))
    return np.hstack(parts)

In [130]:
import torch, torch.nn as nn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

def train_nn(X, y, seed=42):
    torch.manual_seed(seed); np.random.seed(seed)
    idx = np.random.permutation(len(X)); nv = int(0.1 * len(X))
    va, tr = idx[:nv], idx[nv:]
    Xt = torch.tensor(X[tr], dtype=torch.float32); yt = torch.tensor(y[tr], dtype=torch.float32).view(-1, 1)
    Xv = torch.tensor(X[va], dtype=torch.float32).to(dev)
    model = nn.Sequential(nn.Linear(X.shape[1], 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1)).to(dev)
    opt = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, nesterov=True, weight_decay=0.005)
    lossf = nn.MSELoss()
    best, best_state, prev, bad = -1e9, None, -1e9, 0
    for ep in range(200):
        model.train()
        perm = torch.randperm(len(Xt))
        for i in range(0, len(perm), 64):
            b = perm[i:i+64]
            opt.zero_grad()
            lossf(model(Xt[b].to(dev)), yt[b].to(dev)).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            r2 = r2_score(y[va], model(Xv).cpu().numpy().ravel())
            if r2 > best:
                best = r2; best_state = {k: v.clone() for k, v in model.state_dict().items()}
            bad = bad + 1 if abs(r2 - prev) < 1e-4 else 0
            prev = r2
        if bad >= 10: break
    model.load_state_dict(best_state); model.eval()
    return model

def predict(model, X):
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32).to(dev)).cpu().numpy().ravel()

In [131]:
y_berlin = np.log(berlin['price'].values)
results = []
for name, train_df in [
    ('NYC -> Berlin', nyc),
    ('Paris -> Berlin', paris),
    ('NYC+Paris -> Berlin', pd.concat([nyc, paris], ignore_index=True))
]:
    s = make_spec(train_df)
    X = transform(train_df, s)
    y = np.log(train_df['price'].values)
    mu, sd = X.mean(0), X.std(0)
    sd[sd == 0] = 1
    model = train_nn((X - mu) / sd, y)
    p_tr = predict(model, (X - mu) / sd)
    p_b = predict(model, (transform(berlin, s) - mu) / sd)
    results.append({
        'experiment': name,
        'features': X.shape[1],
        'train_MSE': mean_squared_error(y, p_tr),
        'train_R2': r2_score(y, p_tr),
        'berlin_MSE': mean_squared_error(y_berlin, p_b),
        'berlin_R2': r2_score(y_berlin, p_b)
    })
res = pd.DataFrame(results)
print(res)
res.to_csv('/content/gdrive/MyDrive/airbnb_project/cross_city_results.csv', index=False)

            experiment  features  train_MSE  train_R2   berlin_MSE  \
0        NYC -> Berlin       258   0.056498  0.868773  2850.287038   
1      Paris -> Berlin       234   0.059109  0.733041    30.872533   
2  NYC+Paris -> Berlin       271   0.082868  0.743768     0.315344   

     berlin_R2  
0 -8243.979291  
1   -88.304478  
2     0.087811  
